<a href="https://colab.research.google.com/github/LionPaul/retro-game-miner/blob/main/snes/notebooks/05_category_mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests
import os
import time

# ==============================================================================
# PIPELINE CONFIGURATION
# ==============================================================================
CONSOLE_NAME = "snes"
CLIENT_ID = ''
CLIENT_SECRET = ''

DATASETS_DIR = "datasets"
# ADJUSTMENT: Correct chained sequence tracking for Step 05
INPUT_FILENAME = f"4_{CONSOLE_NAME}_games_scored_only.xlsx"   # Reading from Code 4 (Optimized)
OUTPUT_FILENAME = f"5_{CONSOLE_NAME}_games_with_genres.xlsx"   # Generating base number 5

INPUT_PATH = os.path.join(DATASETS_DIR, INPUT_FILENAME)
OUTPUT_PATH = os.path.join(DATASETS_DIR, OUTPUT_FILENAME)
# ==============================================================================

def authenticate_igdb(client_id, client_secret):
    """Generates the OAuth2 Access Token required to consume the Twitch IGDB API."""
    url = f"https://id.twitch.tv/oauth2/token?client_id={client_id}&client_secret={client_secret}&grant_type=client_credentials"
    response = requests.post(url)
    response.raise_for_status()
    return response.json()['access_token']

def fetch_genres_dictionary(headers):
    """Fetches the official translation table of IDs to Genre Names from IGDB."""
    print("📚 Mapping official IGDB genres dictionary...")
    url = "https://api.igdb.com/v4/genres"
    query = "fields id, name; limit 500;"
    response = requests.post(url, headers=headers, data=query)
    if response.status_code == 200:
        # Transforms the list into a lookup dictionary {ID: Name} (e.g., {12: "Role-playing (RPG)"})
        return {item['id']: item['name'] for item in response.json()}
    return {}

def fetch_game_genres_api(game_name, headers, genres_map):
    """Fetches and maps up to 5 genre names for a given game from IGDB."""
    if not game_name or pd.isna(game_name) or str(game_name).strip() == "":
        return [None] * 5

    url = "https://api.igdb.com/v4/games"
    # Query structure targeting the 'genres' array field
    query = f'search "{str(game_name).strip()}"; fields genres; where platforms = (19); limit 1;'

    try:
        response = requests.post(url, headers=headers, data=query)
        if response.status_code == 200:
            data = response.json()
            if data and 'genres' in data[0]:
                genre_ids = data[0]['genres']
                # Translates numeric IDs using our loaded dictionary
                genre_names = [genres_map.get(gid) for gid in genre_ids if gid in genres_map]

                # Normalizes array length to exactly 5 elements padded with None
                genre_names = genre_names[:5]
                while len(genre_names) < 5:
                    genre_names.append(None)
                return genre_names
    except Exception:
        pass
    return [None] * 5

def pipeline_categories_gold(input_path, output_path):
    """
    Extracts up to 5 structured genre categories per game to support advanced BI analysis.
    Acts as the Gold Category Layer of the Medallion Architecture.
    """
    print(f"🏷️ [CODE 05] Starting Category Mapping for console: {CONSOLE_NAME.upper()}")

    # 1. Pipeline gatekeeper check
    if not os.path.exists(input_path):
        print(f"❌ Critical Error: Input file '{input_path}' not found. Please run Notebook 04 first.")
        return

    df = pd.read_excel(input_path)
    total_games = len(df)
    print(f"📊 Dataset loaded successfully. Processing categories for {total_games} pre-filtered games.")

    # 2. Authentication and setup
    token = authenticate_igdb(CLIENT_ID, CLIENT_SECRET)
    headers_igdb = {'Client-ID': CLIENT_ID, 'Authorization': f'Bearer {token}'}
    genres_map = fetch_genres_dictionary(headers_igdb)

    # Conceptual structural vectors for the 5 dimensional columns
    cat1, cat2, cat3, cat4, cat5 = [], [], [], [], []

    print(f"\n⚡ Mining game classifications for {total_games} records...\n")

    for idx, row in df.iterrows():
        # Leverages the clean key T1_Clean validated in previous stages
        name = row['T1_Clean']

        # Requests array of translated categories
        game_genres = fetch_game_genres_api(name, headers_igdb, genres_map)

        # Distributes classifications across separate vectors
        cat1.append(game_genres[0])
        cat2.append(game_genres[1])
        cat3.append(game_genres[2])
        cat4.append(game_genres[3])
        cat5.append(game_genres[4])

        # Dynamic tracking stream log
        active_genres = [g for g in game_genres if g]
        print(f"🏷️ [{idx+1}/{total_games}] {name} -> Categories: {active_genres if active_genres else 'Unclassified'}")
        time.sleep(0.26)

    # 3. Injecting structured features into the DataFrame
    df['Category_Primary'] = cat1
    df['Category_Secondary'] = cat2
    df['Category_Tertiary'] = cat3
    df['Category_Quaternary'] = cat4
    df['Category_Quinary'] = cat5

    # 4. Persistence of the enriched data frame
    df.to_excel(output_path, index=False)
    print(f"\n💾 Enriched Category Layer successfully saved at: {output_path}")
    print("═" * 80)

# Run the category mapping process
pipeline_categories_gold(INPUT_PATH, OUTPUT_PATH)